# TP N°3 : Les Réseaux de Neurones Convolutifs


---

**Objectif :** Résoudre un problème de reconnaissance d'images (chiffres manuscrits MNIST) en utilisant 3 architectures CNN différentes.

## 1. Installations & Imports

In [12]:
#!pip install tensorflow keras

In [13]:
import numpy as np
from keras.datasets import mnist
from keras.utils import to_categorical
from keras import backend as K

#K.set_image_data_format('channels_first')

# Fix random seed for reproducibility
seed = 7
np.random.seed(seed)

from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten
from keras.layers import Conv2D, MaxPooling2D
from keras.models import model_from_json



---
## 2. Chargement et préparation des données

### Section 4a — `get_data_mnist()` (sans normalisation)

In [14]:
def get_data_mnist():
    """Charge et prépare le dataset MNIST (sans normalisation)."""
    # Charger les données
    (X_train, y_train), (X_test, y_test) = mnist.load_data()

    # Reshape : [samples][width][height][channels]
    X_train = X_train.reshape(X_train.shape[0], 28, 28, 1).astype('float32')
    X_test  = X_test.reshape(X_test.shape[0], 28, 28, 1).astype('float32')

    # One-hot encoding des sorties
    y_train = to_categorical(y_train)
    y_test  = to_categorical(y_test)
    num_classes = y_test.shape[1]

    return (X_train, y_train), (X_test, y_test), num_classes


(X_train, y_train), (X_test, y_test), num_classes = get_data_mnist()
print(f"X_train shape : {X_train.shape}")
print(f"X_test  shape : {X_test.shape}")
print(f"Nombre de classes : {num_classes}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
X_train shape : (60000, 28, 28, 1)
X_test  shape : (10000, 28, 28, 1)
Nombre de classes : 10


---
## 3. Small CNN (sans normalisation)

### Section 4b — `small_model()`

Architecture :
- Conv2D 64 filtres 3×3 + ReLU
- Conv2D 32 filtres 3×3 + ReLU
- Flatten
- Dense 10 sorties + Softmax

In [15]:
def small_model():
    """Construit et compile le Small CNN."""
    model = Sequential()
    model.add(Conv2D(64, (3, 3), input_shape=(28, 28, 1), activation='relu'))
    model.add(Conv2D(32, (3, 3), activation='relu'))
    model.add(Flatten())
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(loss='categorical_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])
    return model


model_small = small_model()
model_small.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_5 (Conv2D)               │ (None, 26, 26, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 24, 24, 32)     │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 18432)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 10)             │       184,330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 203,434 (794.66 KB)

 Trainable params: 203,434 (794.66 KB)

 Non-trainable params: 0 (0.00 B)

### Section 4c — Entraînement du Small CNN

**Question 1 :** Instruction d'entraînement

In [16]:
model_small.fit(X_train, y_train,
                validation_data=(X_test, y_test),
                epochs=10,
                batch_size=200)

Epoch 1/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.9294 - loss: 0.5864 - val_accuracy: 0.9741 - val_loss: 0.0842
Epoch 2/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9806 - loss: 0.0623 - val_accuracy: 0.9776 - val_loss: 0.0721
Epoch 3/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9881 - loss: 0.0377 - val_accuracy: 0.9788 - val_loss: 0.0757
Epoch 4/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9905 - loss: 0.0281 - val_accuracy: 0.9789 - val_loss: 0.0785
Epoch 5/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9920 - loss: 0.0240 - val_accuracy: 0.9774 - val_loss: 0.0863
Epoch 6/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9940 - loss: 0.0173 - val_accuracy: 0.9787 - val_loss: 0.0931
Epoch 7/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9953 - loss: 0.0137 - val_accuracy: 0.9784 - val_loss: 0.1075
Epoch 8/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9948 - loss: 0.0160 - val_accuracy: 0

### Section 4c — Évaluation du Small CNN

**Question 2 :** Fonction d'évaluation

In [17]:
def print_model_error_rate(model, X_test, y_test):
    """Évalue le modèle et affiche le taux de succès et d'erreur."""
    scores = model.evaluate(X_test, y_test, verbose=0)
    print("Model score     : %.2f%%" % (scores[1] * 100))
    print("Model error rate: %.2f%%" % (100 - scores[1] * 100))


print(" --- Small CNN ---")
print_model_error_rate(model_small, X_test, y_test)

 --- Small CNN ---
Model score     : 97.71%
Model error rate: 2.29%


---
## 4. Construction des 3 modèles AVEC normalisation

### Section 5.1 — `get_data_mnist()` avec normalisation

Les valeurs pixels sont divisées par 255 pour obtenir des flottants entre 0 et 1.

In [18]:
def get_data_mnist_normalized():
    """Charge et prépare le dataset MNIST avec normalisation (valeurs entre 0 et 1)."""
    (X_train, y_train), (X_test, y_test) = mnist.load_data()

    # Reshape
    X_train = X_train.reshape(X_train.shape[0],28, 28, 1).astype('float32')
    X_test  = X_test.reshape(X_test.shape[0],28, 28, 1).astype('float32')

    # Normalisation : ramener les valeurs dans [0, 1]
    X_train = X_train / 255.0
    X_test  = X_test  / 255.0

    # One-hot encoding
    y_train = to_categorical(y_train)
    y_test  = to_categorical(y_test)
    num_classes = y_test.shape[1]

    return (X_train, y_train), (X_test, y_test), num_classes


(X_train, y_train), (X_test, y_test), num_classes = get_data_mnist_normalized()
print(f"Min pixel value : {X_train.min():.2f} | Max pixel value : {X_train.max():.2f}")

Min pixel value : 0.00 | Max pixel value : 1.00


### Section 5.2 — Small CNN avec normalisation

On réutilise la même architecture `small_model()` mais avec les données normalisées.

In [19]:
model_small_norm = small_model()

model_small_norm.fit(X_train, y_train,
                     validation_data=(X_test, y_test),
                     epochs=10,
                     batch_size=200)

print("-- Small CNN (avec normalisation) ---")
print_model_error_rate(model_small_norm, X_test, y_test)

Epoch 1/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9367 - loss: 0.2215 - val_accuracy: 0.9806 - val_loss: 0.0614
Epoch 2/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9814 - loss: 0.0617 - val_accuracy: 0.9789 - val_loss: 0.0632
Epoch 3/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9868 - loss: 0.0434 - val_accuracy: 0.9860 - val_loss: 0.0429
Epoch 4/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9898 - loss: 0.0327 - val_accuracy: 0.9864 - val_loss: 0.0438
Epoch 5/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9920 - loss: 0.0252 - val_accuracy: 0.9875 - val_loss: 0.0428
Epoch 6/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9938 - loss: 0.0197 - val_accuracy: 0.9873 - val_loss: 0.0449
Epoch 7/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9954 - loss: 0.0149 - val_accuracy: 0.9869 - val_loss: 0.0457
Epoch 8/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9960 - loss: 0.0125 - val_accuracy: 0

Le nouveau taux d'erreur est de 1.38% contre 2.32% sans normalisation.

on peut dire que la normalisation des données est une étape essentielle . Elle améliore significativement les performances même sur un petit réseau comme celui-ci.

### Section 5.3 — Medium CNN

Architecture :
- Conv2D 32 filtres 5×5 + ReLU
- MaxPooling2D 2×2
- Dropout 0.2
- Flatten
- Dense 128 sorties + ReLU
- Dense 10 sorties + Softmax

In [20]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense

def medium_model():
    model = Sequential()
    model.add(Conv2D(32, (5, 5), input_shape=(28, 28, 1), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.2))
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dense(10, activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

model_medium = medium_model()
model_medium.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_9 (Conv2D)               │ (None, 24, 24, 32)     │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 12, 12, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_6 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 592,074 (2.26 MB)

 Trainable params: 592,074 (2.26 MB)

 Non-trainable params: 0 (0.00 B)

In [21]:
(X_train_norm, y_train_norm), (X_test_norm, y_test_norm), num_classes = get_data_mnist_normalized()

model_medium.fit(X_train_norm, y_train_norm, validation_data=(X_test_norm, y_test_norm), epochs=10, batch_size=200)

print("=== Medium CNN ===")
print_model_error_rate(model_medium, X_test_norm, y_test_norm)

Epoch 1/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9304 - loss: 0.2467 - val_accuracy: 0.9774 - val_loss: 0.0717
Epoch 2/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9779 - loss: 0.0733 - val_accuracy: 0.9850 - val_loss: 0.0478
Epoch 3/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9839 - loss: 0.0513 - val_accuracy: 0.9850 - val_loss: 0.0409
Epoch 4/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9872 - loss: 0.0394 - val_accuracy: 0.9878 - val_loss: 0.0356
Epoch 5/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9900 - loss: 0.0323 - val_accuracy: 0.9882 - val_loss: 0.0343
Epoch 6/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9917 - loss: 0.0266 - val_accuracy: 0.9883 - val_loss: 0.0336
Epoch 7/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9930 - loss: 0.0224 - val_accuracy: 0.9891 - val_loss: 0.0307
Epoch 8/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9941 - loss: 0.0185 - val_accuracy: 0.

### Section 5.4 — Large CNN

Architecture :
- Conv2D 30 filtres 5×5 + ReLU
- MaxPooling2D 2×2
- Conv2D 15 filtres 3×3 + ReLU
- Dropout 0.2
- Flatten
- Dense 128 sorties + ReLU
- Dense 50 sorties + ReLU
- Dense 10 sorties + Softmax

In [24]:
def large_model():
    """Construit et compile le Large CNN."""
    model = Sequential()
    model.add(Conv2D(30, (5, 5), input_shape=(28, 28, 1), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Conv2D(15, (3, 3), activation='relu'))
    model.add(Dropout(0.2))
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dense(50,  activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(loss='categorical_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])
    return model


model_large = large_model()
model_large.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_11 (Conv2D)              │ (None, 24, 24, 30)     │           780 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 12, 12, 30)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 10, 10, 15)     │         4,065 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 10, 10, 15)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_7 (Flatten)             │ (None, 1500)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │       192,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 50)             │         6,450 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 10)             │           510 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 203,933 (796.61 KB)

 Trainable params: 203,933 (796.61 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
model_large.fit(X_train, y_train,
                validation_data=(X_test, y_test),
                epochs=10,
                batch_size=200)

print("=== Large CNN ===")
print_model_error_rate(model_large, X_test, y_test)

Epoch 1/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9114 - loss: 0.2923 - val_accuracy: 0.9761 - val_loss: 0.0743
Epoch 2/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9773 - loss: 0.0742 - val_accuracy: 0.9841 - val_loss: 0.0495
Epoch 3/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9838 - loss: 0.0516 - val_accuracy: 0.9893 - val_loss: 0.0350
Epoch 4/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9873 - loss: 0.0414 - val_accuracy: 0.9895 - val_loss: 0.0309
Epoch 5/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9893 - loss: 0.0342 - val_accuracy: 0.9892 - val_loss: 0.0340
Epoch 6/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9912 - loss: 0.0278 - val_accuracy: 0.9907 - val_loss: 0.0278
Epoch 7/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9919 - loss: 0.0245 - val_accuracy: 0.9925 - val_loss: 0.0261
Epoch 8/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9935 - loss: 0.0210 - val_accuracy: 0.

---
## 5. Comparaison des 3 modèles

In [26]:
print("\n========== Comparaison des modèles ==========")
print("\nSmall CNN (sans normalisation) :")
print_model_error_rate(model_small, X_test, y_test)

print("\nSmall CNN (avec normalisation) :")
print_model_error_rate(model_small_norm, X_test, y_test)

print("\nMedium CNN :")
print_model_error_rate(model_medium, X_test, y_test)

print("\nLarge CNN :")
print_model_error_rate(model_large, X_test, y_test)


========== Comparaison des modèles ==========

Small CNN (sans normalisation) :
Model score     : 90.03%
Model error rate: 9.97%

Small CNN (avec normalisation) :
Model score     : 98.45%
Model error rate: 1.55%

Medium CNN :
Model score     : 98.94%
Model error rate: 1.06%

Large CNN :
Model score     : 99.22%
Model error rate: 0.78%


---
## 6. Sauvegarde et chargement des modèles Keras

In [28]:
def save_keras_model(model, filename):
    model_json = model.to_json()
    with open(filename + ".json", "w") as json_file:
        json_file.write(model_json)
    model.save_weights(filename + ".weights.h5")
    print(f"Modèle sauvegardé : {filename}.json + {filename}.weights.h5")

def load_keras_model(filename):
    with open(filename + ".json", 'r') as json_file:
        loaded_model_json = json_file.read()
    loaded_model = model_from_json(loaded_model_json)
    loaded_model.load_weights(filename + ".weights.h5")
    print(f"Modèle chargé depuis : {filename}.json + {filename}.weights.h5")
    return loaded_model

save_keras_model(model_large, "large_cnn")

loaded = load_keras_model("large_cnn")
loaded.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
print("\nVérification du modèle rechargé :")
print_model_error_rate(loaded, X_test, y_test)

Modèle sauvegardé : large_cnn.json + large_cnn.weights.h5
Modèle chargé depuis : large_cnn.json + large_cnn.weights.h5

Vérification du modèle rechargé :
Model score     : 99.22%
Model error rate: 0.78%
